# 邊緣端資料前處理

剃除了以下不適用的感測特徵
*  **膜厚** (屬最終檢驗結果，非即時感測特徵)
*  **三軸振動加速度** 
*  **減速機溫度** 

---

##  一、 原始感測資料輸入與模擬邏輯 (Sensor Data Simulation Logic)

保留的核心特徵依照其物理特性，設計了專屬的動態模擬邏輯，以高度還原真實產線的設備老化與環境變化：

### 1. 環境與設備健康度 (Background / Time-Series)
| 感測特徵 | 欄位名稱 | 單位 | 物理模擬邏輯設計 |
| :--- | :--- | :--- | :--- |
| **濾網壓差** | `filter_diff_pressure_bar` | `bar` | **線性上升**（呈鋸齒波，設定每天早上定時更換濾網並歸零至 0.1 bar）。 |
| **伺服馬達負載** | `servo_torque_load_pct` | `%` | **極緩慢線性遞增**（以 5~6 個月為極限損壞週期來設定微小斜率）。 |
| **空氣壓力** | `air_pressure_bar` | `bar` | **穩定輸出**（設定基準值 2.5 bar，並疊加微小的常態分佈雜訊）。 |
| **環境溫度** | `temperature_c` | `°C` | **週期弦波** 疊加常態分佈雜訊。 |
| **環境濕度** | `humidity_rh` | `%` | **2週期弦波** 疊加常態分佈雜訊。 |

### 2. 製程與系統狀態標記 (Process & System Event)
| 感測特徵 / 標記 | 欄位名稱 | 單位 | 物理 / 系統邏輯設計 |
| :--- | :--- | :--- | :--- |
| **塗料流量** | `paint_flow_ml_min` | `ml/min` | 隨生產批次呈 **微幅線性遞減**。 |
| **噴幅寬度** | `spray_width_mm` | `mm` | 隨生產批次呈 **微幅線性遞減**。 |
| **軌跡追隨誤差** | `path_error_mm` | `mm` | 隨生產批次（機構老化）呈 **微幅線性遞增**。 |
| **資料品質標記** | `data_quality_flag` | `N/A` | **系統清洗後自動賦值**（正常通訊標記為 `normal`，若遇斷線空值並經由前處理補值者，標記為 `interpolated`）。 |

---

## 二、 感測器門檻值定義 (Thresholds Definitions)
為了提供後端 Rule Engine 觸發警報以及 AI 團隊判定品質基準，本模組制訂了以下各項感測輸入的正常 (Normal)、警告 (Warning) 與異常 (Alarm) 判定區間：

| 欄位名稱 (Sensor) | 正常區間 (Normal) | 警告區間 (Warning) | 異常區間 (Alarm) | 異常常見原因 |
| :--- | :--- | :--- | :--- | :--- |
| `filter_diff_pressure_bar` | `0.1 ~ 0.5` | `0.5 ~ 0.7` | `> 0.7` | 濾網嚴重阻塞，需立即更換 |
| `servo_torque_load_pct` | `40.0 ~ 60.0` | `60.0 ~ 75.0` | `> 75.0` | 軸承磨損或機構卡澀風險 |
| `air_pressure_bar` | `2.4 ~ 2.6` | `2.3~2.4` / `2.6~2.7` | `< 2.3` 或 `> 2.7` | 空壓機供氣異常或管路洩漏 |
| `temperature_c` | `20.0 ~ 30.0` | `15.0~20.0` / `30.0~35.0` | `< 15.0` 或 `> 35.0` | 廠務空調失效，影響塗料揮發 |
| `humidity_rh` | `45.0 ~ 65.0` | `35.0~45.0` / `65.0~75.0` | `< 35.0` 或 `> 75.0` | 廠區濕度過高，易造成表面漆霧 |
| `paint_flow_ml_min` | `105.0 ~ 125.0` | `95.0 ~ 105.0` | `< 95.0` 或 `> 125.0` | 噴嘴堵塞 (低) 或 噴嘴嚴重磨損 (高) |
| `spray_width_mm` | `105.0 ~ 125.0` | `95.0 ~ 105.0` | `< 95.0` 或 `> 125.0` | 噴嘴積漆影響扇幅 (低) |
| `path_error_mm` | `0.00 ~ 0.10` | `0.10 ~ 0.15` | `> 0.15` | 手臂減速機老化或 TCP 偏移 |

---

##  三、 資料流清洗與建構步驟 (Data Pipeline Steps)

本模組的 Python 程式邏輯嚴格遵循以下步驟，將底層帶有雜訊的 Raw Data 轉為結構化數據：

###  Step 1: 斷線偵測與空值修復 (Missing Value Imputation)
讀取機台底層傳來的 **1 分鐘 (1 min) 頻率時序 CSV 與批次 CSV 訊號**。系統會自動偵測 `NaN` 網路瞬斷坑洞，使用前值填補法 (Forward Fill) 進行補值，並將該筆資料嚴格標記為 `"interpolated"`，正常通訊者則標記為 `"normal"`。

###  Step 2: 頻率解耦與時間對齊 (Frequency Decoupling)
將感測頻率徹底解耦，分別產出兩種頻率的資料流：
1. **By 時間 (1 min) 時序資料**：每 1 分鐘生成一筆環境與設備健康快照。
2. **By 批次 (Event) 事件資料**：精準計算 50 秒噴塗區間，並賦予 `batch_id`、`batch_start_time` 與 `batch_end_time`。

###  Step 3: 多格式輸出 
同時生成JSON CSV


## 四、 最終產出 (Deliverables)

模組執行後，產出 3 大資料夾（涵蓋 JSON 與 CSV 全格式），皆已完成資料品質保證與遺失值修復，可直接供下游系統串接：

*  **`Station_Combined_json`**
      結合時間與批次的json檔
*  **`timeandbatch_JSON`**
      批次的json檔
      時間的json檔
*  **`Dataprocess_csv`**
      資料清洗後的csv檔


In [3]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

def generate_multi_station_raw_data():
    print("🚀 啟動 [3 站點 10 天份] 機台原始資料 (Raw Data) 擷取...")
    
    OUTPUT_DIR = "0_Raw_Sensor_Data"
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    start_date = datetime(2026, 6, 1)
    days = 10
    stations = ["Station_1", "Station_2", "Station_3"]
    
    # 建立 10 天的完整 1 分鐘時間軸
    end_date = start_date + timedelta(days=days)
    full_time_index = pd.date_range(start=start_date, end=end_date, freq='1min')
    
    # 篩選掉下班時間與午休 (只保留 08:00~11:59 與 13:00~16:59)
    working_hours_mask = full_time_index.hour.isin([8, 9, 10, 11, 13, 14, 15, 16])
    time_index = full_time_index[working_hours_mask]
    n_time_rows = len(time_index)
    
    # ==========================================
    # 1. 產生 By 時間 (1分鐘1筆) 的環境 Raw Data (3個站點)
    # ==========================================
    for station in stations:
        # 針對不同站點給予些微的環境差異
        temp_base = 25.0 if station == "Station_1" else (25.5 if station == "Station_2" else 24.8)
        
        df_time = pd.DataFrame({
            "timestamp": time_index.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "station": station,
            "temperature_c": np.round(np.random.normal(temp_base, 0.5, n_time_rows), 2),
            "humidity_rh": np.round(np.random.normal(55.0, 1.0, n_time_rows), 2),
            "filter_diff_pressure_bar": np.round(np.linspace(0.1, 0.4, n_time_rows) + np.random.normal(0, 0.005, n_time_rows), 3),
            "servo_torque_load_pct": np.round(np.random.normal(45.0, 1.0, n_time_rows), 2),
            "air_pressure_bar": np.round(np.random.normal(2.5, 0.02, n_time_rows), 2)
        })
        
        # ⚠️ 【製造瑕疵】：模擬網路瞬斷，隨機挖空 3% 的濾網壓差與溫度資料 (變成 NaN)
        nan_idx_filter = np.random.choice(n_time_rows, size=int(n_time_rows * 0.03), replace=False)
        nan_idx_temp = np.random.choice(n_time_rows, size=int(n_time_rows * 0.03), replace=False)
        df_time.loc[nan_idx_filter, "filter_diff_pressure_bar"] = np.nan
        df_time.loc[nan_idx_temp, "temperature_c"] = np.nan
        
        # 存檔
        file_path = os.path.join(OUTPUT_DIR, f"Raw_{station}_TimeSeries.csv")
        df_time.to_csv(file_path, index=False)
        print(f"✅ 已產出: {file_path} (共 {n_time_rows} 筆，包含模擬斷線空值)")

    # ==========================================
    # 2. 產生 By 批次 (Event) 的生產履歷 Raw Data
    # ==========================================
    batch_records = { "Station_1": [], "Station_2": [], "Station_3": [] }
    batch_count = 1
    
    for day in range(days):
        current_date = start_date + timedelta(days=day)
        
        # 上午班 (08:00 - 12:00) 與 下午班 (13:00 - 17:00)
        for start_hour, end_hour in [(8, 12), (13, 17)]:
            curr_time = current_date.replace(hour=start_hour, minute=0, second=0)
            end_time = current_date.replace(hour=end_hour, minute=0, second=0)
            
            while curr_time < end_time:
                batch_id = f"B-{current_date.strftime('%Y%m%d')}-{batch_count:04d}"
                
                # 計算三個站點的接力時間 (每站噴塗50秒，傳送120秒)
                t1_start = curr_time
                t2_start = t1_start + timedelta(seconds=170)
                t3_start = t2_start + timedelta(seconds=170)
                
                for st, t_start in zip(stations, [t1_start, t2_start, t3_start]):
                    # 確保最後一站完成時沒有超過下班時間
                    if t_start + timedelta(seconds=50) <= end_time:
                        batch_records[st].append({
                            "event_type": "batch_completed",
                            "station": st,
                            "batch_id": batch_id,
                            "batch_start_time": t_start.strftime("%Y-%m-%dT%H:%M:%SZ"),
                            "batch_end_time": (t_start + timedelta(seconds=50)).strftime("%Y-%m-%dT%H:%M:%SZ"),
                            # ⚠️ 原始資料沒有 data_quality_flag
                            "paint_flow_ml_min": round(np.random.normal(120.0, 1.5), 2),
                            "spray_width_mm": round(np.random.normal(118.0, 1.0), 2),
                            "path_error_mm": round(abs(np.random.normal(0.05, 0.01)), 3)
                        })
                        
                batch_count += 1
                curr_time += timedelta(minutes=3) # 每 3 分鐘投料一次

    for station in stations:
        df_batch = pd.DataFrame(batch_records[station])
        file_path = os.path.join(OUTPUT_DIR, f"Raw_{station}_Batch.csv")
        df_batch.to_csv(file_path, index=False)
        print(f"✅ 已產出: {file_path} (共 {len(df_batch)} 筆完工紀錄)")

    print(f"\n🎉 完美！所有原始資料皆已存入 `{OUTPUT_DIR}` 資料夾中。")

# 執行產出
generate_multi_station_raw_data()

🚀 啟動 [3 站點 10 天份] 機台原始資料 (Raw Data) 擷取...
✅ 已產出: 0_Raw_Sensor_Data\Raw_Station_1_TimeSeries.csv (共 4800 筆，包含模擬斷線空值)
✅ 已產出: 0_Raw_Sensor_Data\Raw_Station_2_TimeSeries.csv (共 4800 筆，包含模擬斷線空值)
✅ 已產出: 0_Raw_Sensor_Data\Raw_Station_3_TimeSeries.csv (共 4800 筆，包含模擬斷線空值)
✅ 已產出: 0_Raw_Sensor_Data\Raw_Station_1_Batch.csv (共 1600 筆完工紀錄)
✅ 已產出: 0_Raw_Sensor_Data\Raw_Station_2_Batch.csv (共 1580 筆完工紀錄)
✅ 已產出: 0_Raw_Sensor_Data\Raw_Station_3_Batch.csv (共 1560 筆完工紀錄)

🎉 完美！所有原始資料皆已存入 `0_Raw_Sensor_Data` 資料夾中。


In [1]:
import pandas as pd
import json
import os

def run_data_pipeline_ultimate():
    print("🚀 啟動邊緣端資料清洗管線 (支援 JSON & CSV 全格式輸出)...")
    
    INPUT_DIR = "0_Raw_Sensor_Data"
    OUTPUT_DIR = "sprayline_final_deliverables"
    
    # 幫資料夾分門別類，這就是專業的 Data Pipeline 產出結構
    API_JSON_DIR = os.path.join(OUTPUT_DIR, "1_for_Backend_API_JSON")
    DB_JSON_DIR = os.path.join(OUTPUT_DIR, "2_for_Database_NoSQL_JSON")
    DB_CSV_DIR = os.path.join(OUTPUT_DIR, "3_for_Database_SQL_CSV")

    for d in [API_JSON_DIR, DB_JSON_DIR, DB_CSV_DIR]:
        if not os.path.exists(d):
            os.makedirs(d)

    stations = ["Station_1", "Station_2", "Station_3"]

    for station in stations:
        print(f"\n⚙️ 正在清洗 {station} 的原始資料...")

        # =========================================
        # 1. 讀取並清洗 Time-Series
        # =========================================
        ts_file = os.path.join(INPUT_DIR, f"Raw_{station}_TimeSeries.csv")
        df_ts = pd.read_csv(ts_file)

        df_ts['data_quality_flag'] = "normal"
        has_nan_mask = df_ts.isna().any(axis=1)
        df_ts.loc[has_nan_mask, 'data_quality_flag'] = "interpolated"
        df_ts.fillna(method='ffill', inplace=True)
        df_ts.fillna(method='bfill', inplace=True)
        
        ts_data = df_ts.to_dict(orient='records')

        # =========================================
        # 2. 讀取並清洗 Batch
        # =========================================
        batch_file = os.path.join(INPUT_DIR, f"Raw_{station}_Batch.csv")
        df_batch = pd.read_csv(batch_file)

        df_batch['data_quality_flag'] = "normal"
        batch_data = df_batch.to_dict(orient='records')

        # =========================================
        # 3. 輸出：JSON 格式 (給 API 與 NoSQL)
        # =========================================
        combined_data = {"time_series": ts_data, "batches": batch_data}
        with open(os.path.join(API_JSON_DIR, f"{station}_Combined.json"), 'w', encoding='utf-8') as f:
            json.dump(combined_data, f, indent=2, ensure_ascii=False)

        with open(os.path.join(DB_JSON_DIR, f"{station}_TimeSeries.json"), 'w', encoding='utf-8') as f:
            json.dump(ts_data, f, indent=2, ensure_ascii=False)

        with open(os.path.join(DB_JSON_DIR, f"{station}_BatchLog.json"), 'w', encoding='utf-8') as f:
            json.dump(batch_data, f, indent=2, ensure_ascii=False)

        # =========================================
        # 4. 輸出：CSV 格式 (給 SQL 資料庫與 AI 團隊看表用)
        # =========================================
        # 因為 df_ts 跟 df_batch 已經被我們洗乾淨了，直接 to_csv 吐出去就好！
        ts_csv_path = os.path.join(DB_CSV_DIR, f"{station}_TimeSeries_Cleaned.csv")
        df_ts.to_csv(ts_csv_path, index=False, encoding='utf-8')
        
        batch_csv_path = os.path.join(DB_CSV_DIR, f"{station}_BatchLog_Cleaned.csv")
        df_batch.to_csv(batch_csv_path, index=False, encoding='utf-8')

        print(f"  └─ 成功產出 3 份 JSON 與 2 份 CSV！(修復了 {has_nan_mask.sum()} 筆斷線數據)")

    print(f"\n🎉 恭喜！全線資料前處理完成！請至 `{OUTPUT_DIR}` 查看所有格式成果。")

# 執行
run_data_pipeline_ultimate()

🚀 啟動邊緣端資料清洗管線 (支援 JSON & CSV 全格式輸出)...

⚙️ 正在清洗 Station_1 的原始資料...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_39620\2345352564.py:34: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ts.fillna(method='ffill', inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_39620\2345352564.py:35: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ts.fillna(method='bfill', inplace=True)


  └─ 成功產出 3 份 JSON 與 2 份 CSV！(修復了 285 筆斷線數據)

⚙️ 正在清洗 Station_2 的原始資料...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_39620\2345352564.py:34: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ts.fillna(method='ffill', inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_39620\2345352564.py:35: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ts.fillna(method='bfill', inplace=True)


  └─ 成功產出 3 份 JSON 與 2 份 CSV！(修復了 281 筆斷線數據)

⚙️ 正在清洗 Station_3 的原始資料...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_39620\2345352564.py:34: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ts.fillna(method='ffill', inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_39620\2345352564.py:35: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ts.fillna(method='bfill', inplace=True)


  └─ 成功產出 3 份 JSON 與 2 份 CSV！(修復了 284 筆斷線數據)

🎉 恭喜！全線資料前處理完成！請至 `sprayline_final_deliverables` 查看所有格式成果。
